<a href="https://colab.research.google.com/github/Thanwarin/robot-webots/blob/main/EfficientNetB3_%2B_FER201.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook Summary:

- Implements facial emotion recognition using an EfficientNetB3 CNN.

- Uses a two-stage training strategy:

  - Stage 1: Freeze backbone, train classifier layers.

  - Stage 2: Fine-tune entire network for better performance.

- Applies data augmentation and handles class imbalance with class weights.

- Saves intermediate and final models to Google Drive.

- Note: This code is experimental and may take a long time to run; it’s intended for testing and prototyping.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# =====================================
# Imports
# =====================================
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import os
import glob

print("TF version:", tf.__version__)

# =====================================
# Paths & Config
# =====================================
IMG_SIZE = 224
BATCH_SIZE = 64
NUM_CLASSES = 7
EPOCHS_STAGE1 = 20
EPOCHS_STAGE2 = 15

SAVE_PATH = "/content/drive/MyDrive/ml_models"
os.makedirs(SAVE_PATH, exist_ok=True)

train_dir = "/content/drive/MyDrive/emotion_dataset/train"
val_dir   = "/content/drive/MyDrive/emotion_dataset/val"

# =====================================
# Compute Class Weights
# =====================================
labels = []
for c in range(NUM_CLASSES):
    labels += [c] * len(glob.glob(f"{train_dir}/{c}/*"))

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=labels
)
class_weight_dict = {i: w for i, w in enumerate(class_weights)}
print("Class weights:", class_weight_dict)

# =====================================
# Data Augmentation
# =====================================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.3,
    horizontal_flip=True,
    brightness_range=[0.7,1.3]
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode="grayscale",
    class_mode="categorical",
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_gen = val_datagen.flow_from_directory(
    val_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode="grayscale",
    class_mode="categorical",
    batch_size=BATCH_SIZE,
    shuffle=False
)

# =====================================
# Build EfficientNetB3 Model
# =====================================
def build_model():
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1))

    # Convert grayscale → 3 channels
    x = layers.Conv2D(3, (3,3), padding="same")(inputs)

    # EfficientNetB3 backbone
    base = tf.keras.applications.EfficientNetB3(
        include_top=False,
        weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base.trainable = False  # Stage 1: freeze

    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(inputs, outputs)
    return model

model = build_model()
model.summary()

# =====================================
# Compile Stage 1
# =====================================
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

# =====================================
# Callbacks Stage 1
# =====================================
ckpt_stage1 = os.path.join(SAVE_PATH, "efficientnetb3_stage1.h5")
callbacks_stage1 = [
    EarlyStopping(patience=6, restore_best_weights=True),
    ReduceLROnPlateau(factor=0.3, patience=3, min_lr=1e-6),
    ModelCheckpoint(ckpt_stage1, save_best_only=True, monitor="val_accuracy")
]

# =====================================
# Train Stage 1 (Freeze Backbone)
# =====================================
history1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_STAGE1,
    class_weight=class_weight_dict,
    callbacks=callbacks_stage1
)

# =====================================
# Stage 2: Fine-Tune Entire EfficientNetB3
# =====================================
model.get_layer(index=2).trainable = True  # unfreeze base model

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

ckpt_stage2 = os.path.join(SAVE_PATH, "efficientnetb3_stage2.h5")
callbacks_stage2 = [
    EarlyStopping(patience=8, restore_best_weights=True),
    ReduceLROnPlateau(factor=0.3, patience=3, min_lr=1e-7),
    ModelCheckpoint(ckpt_stage2, save_best_only=True, monitor="val_accuracy")
]

history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_STAGE2,
    class_weight=class_weight_dict,
    callbacks=callbacks_stage2
)

# =====================================
# Save Final Model
# =====================================
final_model_path = os.path.join(SAVE_PATH, "emotion_efficientnetb3_final.h5")
model.save(final_model_path)
print("Saved model at:", final_model_path)


TF version: 2.19.0
Class weights: {0: np.float64(1.0093864468864469), 1: np.float64(9.471535982814178), 2: np.float64(0.9966094032549728), 3: np.float64(0.5606205098861975), 4: np.float64(0.8569484936831876), 5: np.float64(1.3190725504861631), 6: np.float64(0.8392500237936613)}
Found 8838 images belonging to 7 classes.
Found 0 images belonging to 7 classes.
43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 3)    │            30 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb3 (Functional)     │ (None, 7, 7, 1536)     │    10,783,535 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1536)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1536)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       393,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,178,836 (42.64 MB)

 Trainable params: 395,301 (1.51 MB)

 Non-trainable params: 10,783,535 (41.14 MB)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 0s 31s/step - accuracy: 0.1201 - loss: 2.0257 

ValueError: The PyDataset has length 0

In [ ]:
train_gen

In [ ]:
import glob

print("Train class 0 images:", len(glob.glob("/content/drive/MyDrive/emotion_dataset/train/0/*")))
print("Val class 0 images:", len(glob.glob("/content/drive/MyDrive/emotion_dataset/val/0/*")))


Train class 0 images: 3995
Val class 0 images: 467


In [ ]:
# =====================================
# Imports
# =====================================
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os
import glob
import numpy as np

print("TF version:", tf.__version__)

# =====================================
# Paths & Config
# =====================================
IMG_SIZE = 300  # EfficientNetB3 default input
BATCH_SIZE = 32
NUM_CLASSES = 7
EPOCHS_STAGE1 = 20
EPOCHS_STAGE2 = 15

SAVE_PATH = "/content/drive/MyDrive/ml_models"
os.makedirs(SAVE_PATH, exist_ok=True)

train_dir = "/content/drive/MyDrive/emotion_dataset/train"
val_dir   = "/content/drive/MyDrive/emotion_dataset/val"

# =====================================
# Data Augmentation
# =====================================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.3,
    horizontal_flip=True,
    brightness_range=[0.7,1.3],
    fill_mode="nearest"
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode="rgb",  # convert grayscale → 3 channels
    class_mode="categorical",
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_gen = val_datagen.flow_from_directory(
    val_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode="rgb",
    class_mode="categorical",
    batch_size=BATCH_SIZE,
    shuffle=False
)

# =====================================
# Build EfficientNetB3 Model
# =====================================
base = tf.keras.applications.EfficientNetB3(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base.trainable = False  # Stage 1 freeze

x = base.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = models.Model(base.input, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

model.summary()

# =====================================
# Callbacks Stage 1
# =====================================
ckpt_stage1 = os.path.join(SAVE_PATH, "efficientnetb3_stage1.h5")
callbacks_stage1 = [
    EarlyStopping(patience=6, restore_best_weights=True),
    ReduceLROnPlateau(factor=0.3, patience=3, min_lr=1e-6),
    ModelCheckpoint(ckpt_stage1, save_best_only=True, monitor="val_accuracy")
]

# =====================================
# Train Stage 1 (Freeze Backbone)
# =====================================
history1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_STAGE1,
    callbacks=callbacks_stage1
)

# =====================================
# Stage 2: Fine-Tune Entire EfficientNetB3
# =====================================
base.trainable = True

# optionally freeze first few layers (say 50%) if needed:
# for layer in base.layers[:len(base.layers)//2]:
#     layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

ckpt_stage2 = os.path.join(SAVE_PATH, "efficientnetb3_stage2.h5")
callbacks_stage2 = [
    EarlyStopping(patience=8, restore_best_weights=True),
    ReduceLROnPlateau(factor=0.3, patience=3, min_lr=1e-7),
    ModelCheckpoint(ckpt_stage2, save_best_only=True, monitor="val_accuracy")
]

history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_STAGE2,
    callbacks=callbacks_stage2
)

# =====================================
# Save Final Model
# =====================================
final_model_path = os.path.join(SAVE_PATH, "emotion_efficientnetb3_final.h5")
model.save(final_model_path)
print("Saved model at:", final_model_path)


TF version: 2.19.0
Found 28709 images belonging to 7 classes.
Found 3589 images belonging to 7 classes.


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 300, 300,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 300, 300,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 300, 300,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 300, 300,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 301, 301,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 150, 150,  │      1,080 │ stem_conv_pad[0]… │
│                     │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 150, 150,  │        160 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 150, 150,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 150, 150,  │        360 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 150, 150,  │        160 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 150, 150,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 40)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 40)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 10)  │        410 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 40)  │        440 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 150, 150,  │          0 │ block1a_activati… │
│ (Multiply)          │ 40)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 150, 150,  │        960 │ block1a_se_excit

 Total params: 11,178,806 (42.64 MB)

 Trainable params: 395,271 (1.51 MB)

 Non-trainable params: 10,783,535 (41.14 MB)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
898/898 ━━━━━━━━━━━━━━━━━━━━ 0s 17s/step - accuracy: 0.2240 - loss: 1.8739 

898/898 ━━━━━━━━━━━━━━━━━━━━ 17254s 19s/step - accuracy: 0.2240 - loss: 1.8739 - val_accuracy: 0.2494 - val_loss: 1.8433 - learning_rate: 0.0010
Epoch 2/20
898/898 ━━━━━━━━━━━━━━━━━━━━ 9128s 10s/step - accuracy: 0.2514 - loss: 1.8454 - val_accuracy: 0.2494 - val_loss: 1.8471 - learning_rate: 0.0010
Epoch 3/20
898/898 ━━━━━━━━━━━━━━━━━━━━ 9291s 10s/step - accuracy: 0.2498 - loss: 1.8446 - val_accuracy: 0.2494 - val_loss: 1.8437 - learning_rate: 0.0010
Epoch 4/20
 74/898 ━━━━━━━━━━━━━━━━━━━━ 2:12:01 10s/step - accuracy: 0.2505 - loss: 1.8539

In [ ]:
print()